# XGBoost From Scratch: What the Paper Actually Adds Over Vanilla Gradient Boosting

**Claim:** the difference between classic gradient boosting (Friedman, 2001) and XGBoost (Chen & Guestrin, 2016) is one equation — using the second-order Taylor expansion of the loss instead of just the first-order gradient.

This notebook implements both from scratch, verifies the second-order version against the real `xgboost` library, and benchmarks both on the Titanic dataset (891 real passenger records), ending with a SHAP feature-importance pass.

**The math:**

Vanilla gradient boosting fits a tree to the negative gradient only (first-order). XGBoost instead minimizes a second-order Taylor expansion of the loss around the current prediction:

`L(y, ŷ+f(x)) ≈ L(y, ŷ) + g·f(x) + ½h·f(x)²`

This has a closed-form optimal leaf weight — no line search needed:

`w* = -G_j / (H_j + λ)`

where `G_j` and `H_j` are the summed gradients and Hessians of the training points that land in leaf `j`, and `λ` is an L2 penalty. The same second-order expansion also gives the exact split-quality formula XGBoost uses to grow trees:

`gain = 0.5 · [GL²/(HL+λ) + GR²/(HR+λ) − G²/(H+λ)] − γ`

`λ` isn't a blind regularization knob here — it directly discounts leaf confidence when `H_j` (how much curvature/certainty the data gives you in that leaf) is small, which is exactly why it matters most on small-sample leaves.


## Data: real Titanic passenger records (891 rows, from the public datasciencedojo/datasets CSV)

In [1]:
"""Shared Titanic preprocessing for all models in this repo."""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


def load_titanic(path="titanic.csv", seed=42):
    df = pd.read_csv(path)

    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())

    df["Sex"] = (df["Sex"] == "male").astype(int)
    df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2})

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
                "Embarked", "FamilySize", "IsAlone"]
    X = df[features].values.astype(np.float64)
    y = df["Survived"].values.astype(np.float64)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    return X_train, X_test, y_train, y_test, features


## Implementation: VanillaGBM (first-order) and XGBScratch (second-order)

Both use the identical boosting loop and logistic loss. The only difference is the split criterion and leaf-weight formula — exactly the one equation the post claims it is.

In [2]:
"""
Gradient boosting from scratch, two ways, to make the XGBoost paper's core
contribution concrete: it's one equation.

Both models boost on the logistic loss for binary classification:
  L(y, p) = -[y*log(p) + (1-y)*log(1-p)],  p = sigmoid(raw_score)
  gradient g = p - y
  hessian  h = p * (1 - p)

VANILLA (first-order, classic Friedman-style GBM):
  Fit a regression tree to the negative gradient (-g), splitting on plain
  variance reduction (MSE), same as fitting any regression target.
  Leaf value = mean(-g) in that leaf. No second-order information used at
  all -- this is exactly gradient descent in function space.

XGBOOST-STYLE (second-order):
  Use the SAME boosting loop, but:
    1. split selection uses the exact-greedy gain formula derived from the
       second-order Taylor expansion of the loss:
         gain = 0.5 * [ GL^2/(HL+lam) + GR^2/(HR+lam) - G^2/(H+lam) ] - gamma
    2. leaf value uses the closed-form optimum:
         w* = -G / (H + lam)
  This is the one mathematical difference the post claims it is.
"""

import numpy as np
from sklearn.tree import DecisionTreeRegressor


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


# vanilla --
class VanillaGBM:
    """First-order gradient boosting: fit a plain regression tree to -gradient."""

    def __init__(self, n_estimators=50, max_depth=3, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.lr = learning_rate
        self.trees = []
        self.base_score = 0.0

    def fit(self, X, y):
        p = np.full(len(y), y.mean())
        self.base_score = float(np.log(p[0] / (1 - p[0])))  # logit of base rate
        raw = np.full(len(y), self.base_score)

        for _ in range(self.n_estimators):
            p = sigmoid(raw)
            neg_grad = y - p                       # -g, g = p - y
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, neg_grad)                  # plain MSE split, no Hessian
            update = tree.predict(X)
            raw += self.lr * update
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        raw = np.full(len(X), self.base_score)
        for tree in self.trees:
            raw += self.lr * tree.predict(X)
        return sigmoid(raw)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


# xgboost-style --
class _XGBNode:
    __slots__ = ("feature", "threshold", "left", "right", "weight", "is_leaf")


class _XGBTree:
    """Exact-greedy regression tree split on the second-order gain formula."""

    def __init__(self, max_depth, lam, gamma, min_child_weight):
        self.max_depth = max_depth
        self.lam = lam
        self.gamma = gamma
        self.min_child_weight = min_child_weight
        self.root = None

    def _leaf_weight(self, g, h):
        return -g.sum() / (h.sum() + self.lam)

    def _best_split(self, X, g, h):
        best_gain, best_feat, best_thr = 0.0, None, None
        G, H = g.sum(), h.sum()
        n_features = X.shape[1]
        for f in range(n_features):
            order = np.argsort(X[:, f])
            xf, gf, hf = X[order, f], g[order], h[order]
            GL = np.cumsum(gf)[:-1]
            HL = np.cumsum(hf)[:-1]
            GR = G - GL
            HR = H - HL
            valid = (HL >= self.min_child_weight) & (HR >= self.min_child_weight)
            if not np.any(valid):
                continue
            gain = 0.5 * (GL**2 / (HL + self.lam) + GR**2 / (HR + self.lam)
                           - G**2 / (H + self.lam)) - self.gamma
            gain = np.where(valid, gain, -np.inf)
            # only split between distinct consecutive x-values
            distinct = xf[1:] != xf[:-1]
            gain = np.where(distinct, gain, -np.inf)
            i = np.argmax(gain)
            if gain[i] > best_gain:
                best_gain, best_feat = gain[i], f
                best_thr = (xf[i] + xf[i + 1]) / 2.0
        return best_feat, best_thr, best_gain

    def _build(self, X, g, h, depth):
        node = _XGBNode()
        if depth >= self.max_depth or len(g) < 2 * self.min_child_weight:
            node.is_leaf = True
            node.weight = self._leaf_weight(g, h)
            return node

        feat, thr, gain = self._best_split(X, g, h)
        if feat is None or gain <= 0:
            node.is_leaf = True
            node.weight = self._leaf_weight(g, h)
            return node

        mask = X[:, feat] <= thr
        node.is_leaf = False
        node.feature, node.threshold = feat, thr
        node.left = self._build(X[mask], g[mask], h[mask], depth + 1)
        node.right = self._build(X[~mask], g[~mask], h[~mask], depth + 1)
        return node

    def fit(self, X, g, h):
        self.root = self._build(X, g, h, depth=0)
        return self

    def _predict_one(self, x, node):
        while not node.is_leaf:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node.weight

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])


class XGBScratch:
    """Second-order gradient boosting -- the actual XGBoost split/leaf rule."""

    def __init__(self, n_estimators=50, max_depth=3, learning_rate=0.1,
                 lam=1.0, gamma=0.0, min_child_weight=1.0):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.lr = learning_rate
        self.lam = lam
        self.gamma = gamma
        self.min_child_weight = min_child_weight
        self.trees = []
        self.base_score = 0.0

    def fit(self, X, y):
        p0 = y.mean()
        self.base_score = float(np.log(p0 / (1 - p0)))
        raw = np.full(len(y), self.base_score)

        for _ in range(self.n_estimators):
            p = sigmoid(raw)
            g = p - y                      # gradient of logloss
            h = p * (1 - p)                # hessian of logloss
            tree = _XGBTree(self.max_depth, self.lam, self.gamma, self.min_child_weight)
            tree.fit(X, g, h)
            raw += self.lr * tree.predict(X)
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        raw = np.full(len(X), self.base_score)
        for tree in self.trees:
            raw += self.lr * tree.predict(X)
        return sigmoid(raw)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


## Verification, models, and benchmark

First, a controlled sanity check: run `XGBScratch` for a single boosting round and compare its predicted probabilities against the real `xgboost` library with matching hyperparameters. Then the full comparison: single tree, random forest, vanilla GBM, XGBoost-from-scratch (default and tuned), and the real library for an external reference point.

In [3]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score
import xgboost as xgb

X_train, X_test, y_train, y_test, feature_names = load_titanic()
print(f"Train: {X_train.shape}, Test: {X_test.shape}, features: {feature_names}\n")

# 1. SANITY ===
# Verify XGBScratch's split/leaf math against real xgboost on a controlled,
# tiny, deterministic setting: ONE boosting round, no shrinkage, generous
# lambda/gamma defaults matched exactly, so the only thing that can differ
# is correctness of the from-scratch tree-growing logic itself.
print("=" * 60)
print("Sanity check: XGBScratch vs real xgboost, 1 round, max_depth=2")
print("=" * 60)

my_1round = XGBScratch(n_estimators=1, max_depth=2, learning_rate=1.0,
                        lam=1.0, gamma=0.0, min_child_weight=1.0)
my_1round.fit(X_train, y_train)
my_pred = my_1round.predict_proba(X_train)

real_1round = xgb.XGBClassifier(
    n_estimators=1, max_depth=2, learning_rate=1.0,
    reg_lambda=1.0, gamma=0.0, min_child_weight=1.0,
    base_score=0.5, tree_method="exact", objective="binary:logistic"
)
real_1round.fit(X_train, y_train)
real_pred = real_1round.predict_proba(X_train)[:, 1]

corr = np.corrcoef(my_pred, real_pred)[0, 1]
mean_abs_diff = np.mean(np.abs(my_pred - real_pred))
print(f"Correlation between scratch and real xgboost predicted probabilities: {corr:.4f}")
print(f"Mean abs difference in predicted probability: {mean_abs_diff:.4f}")
print("(Not expected to be bit-identical -- xgboost uses base_score=0.5 by")
print(" default and histogram approximations even in 'exact' mode at the C++")
print(" level, plus float32 internals. High correlation + small diff confirms")
print(" the gain/leaf-weight formulas are implemented correctly.)\n")

# 2. MODELS ===
results = {}

# --- single tree ---
tree = DecisionTreeClassifier(max_depth=4, random_state=0)
tree.fit(X_train, y_train)
results["Single tree (depth=4)"] = accuracy_score(y_test, tree.predict(X_test))

# --- random forest, 100 trees ---
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=0)
rf.fit(X_train, y_train)
results["Random Forest (100 trees)"] = accuracy_score(y_test, rf.predict(X_test))

# --- vanilla GBM from scratch (first-order) ---
vgbm = VanillaGBM(n_estimators=100, max_depth=3, learning_rate=0.1)
vgbm.fit(X_train, y_train)
results["Vanilla GBM from scratch (1st-order)"] = accuracy_score(y_test, vgbm.predict(X_test))

# --- XGBoost from scratch (second-order), untuned defaults ---
xgb_scratch_default = XGBScratch(n_estimators=100, max_depth=3, learning_rate=0.1,
                                  lam=1.0, gamma=0.0, min_child_weight=1.0)
xgb_scratch_default.fit(X_train, y_train)
results["XGBoost from scratch (2nd-order, default)"] = accuracy_score(
    y_test, xgb_scratch_default.predict(X_test))

# --- XGBoost from scratch, tuned via grid search on lambda/depth/lr ---
print("Grid-searching XGBScratch hyperparameters (this takes a minute)...")
best_acc, best_params = -1, None
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)
for max_depth in [2, 3, 4]:
    for lam in [0.1, 1.0, 5.0]:
        for lr in [0.05, 0.1, 0.2]:
            fold_accs = []
            for tr_idx, val_idx in cv.split(X_train, y_train):
                m = XGBScratch(n_estimators=100, max_depth=max_depth,
                                learning_rate=lr, lam=lam, gamma=0.0,
                                min_child_weight=1.0)
                m.fit(X_train[tr_idx], y_train[tr_idx])
                fold_accs.append(accuracy_score(y_train[val_idx], m.predict(X_train[val_idx])))
            mean_acc = np.mean(fold_accs)
            if mean_acc > best_acc:
                best_acc, best_params = mean_acc, dict(max_depth=max_depth, lam=lam, lr=lr)

print(f"Best CV params: {best_params} (CV acc {best_acc:.4f})\n")
xgb_tuned = XGBScratch(n_estimators=100, max_depth=best_params["max_depth"],
                        learning_rate=best_params["lr"], lam=best_params["lam"],
                        gamma=0.0, min_child_weight=1.0)
xgb_tuned.fit(X_train, y_train)
results["XGBoost from scratch (2nd-order, tuned)"] = accuracy_score(y_test, xgb_tuned.predict(X_test))

# --- real xgboost library, tuned, for an external reference point ---
real_xgb = xgb.XGBClassifier(
    n_estimators=100, max_depth=best_params["max_depth"], learning_rate=best_params["lr"],
    reg_lambda=best_params["lam"], eval_metric="logloss", random_state=0
)
real_xgb.fit(X_train, y_train)
results["Real xgboost library (same tuned params)"] = accuracy_score(y_test, real_xgb.predict(X_test))

print("=" * 60)
print("RESULTS on held-out Titanic test set (20%, stratified)")
print("=" * 60)
for name, acc in results.items():
    print(f"  {name:<42} {acc*100:5.1f}%")

import json
with open("results.json", "w") as f:
    json.dump({"results": results, "best_params": best_params}, f, indent=2)
print("\nSaved results.json")


Train: (712, 9), Test: (179, 9), features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']

Sanity check: XGBScratch vs real xgboost, 1 round, max_depth=2
Correlation between scratch and real xgboost predicted probabilities: 1.0000
Mean abs difference in predicted probability: 0.0089
(Not expected to be bit-identical -- xgboost uses base_score=0.5 by
 default and histogram approximations even in 'exact' mode at the C++
 level, plus float32 internals. High correlation + small diff confirms
 the gain/leaf-weight formulas are implemented correctly.)



Grid-searching XGBScratch hyperparameters (this takes a minute)...


Best CV params: {'max_depth': 4, 'lam': 0.1, 'lr': 0.1} (CV acc 0.8357)



RESULTS on held-out Titanic test set (20%, stratified)
  Single tree (depth=4)                       78.8%
  Random Forest (100 trees)                   79.3%
  Vanilla GBM from scratch (1st-order)        80.4%
  XGBoost from scratch (2nd-order, default)   80.4%
  XGBoost from scratch (2nd-order, tuned)     80.4%
  Real xgboost library (same tuned params)    79.9%

Saved results.json


## SHAP feature analysis

Using the real `xgboost` library (fit with the tuned hyperparameters found above) since `shap.TreeExplainer` is built for standard tree libraries. The question: does removing the lowest-impact feature cost any accuracy?

In [4]:
import json
import numpy as np
import shap
import xgboost as xgb
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test, feature_names = load_titanic()

best_params = {"max_depth": 4, "lam": 0.1, "lr": 0.1}  # from the grid search above

model = xgb.XGBClassifier(
    n_estimators=100, max_depth=best_params["max_depth"], learning_rate=best_params["lr"],
    reg_lambda=best_params["lam"], eval_metric="logloss", random_state=0
)
model.fit(X_train, y_train)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
ranking = sorted(zip(feature_names, mean_abs_shap), key=lambda t: -t[1])

print("Mean |SHAP value| per feature (impact on model output):")
for name, val in ranking:
    print(f"  {name:<12} {val:.4f}")

# Keep only features with non-negligible SHAP impact, refit, compare.
keep = [name for name, val in ranking if val > 0.02]
keep_idx = [feature_names.index(f) for f in keep]
dropped = [f for f in feature_names if f not in keep]
print(f"\nDropping low-impact features: {dropped}")
print(f"Keeping: {keep}")

model_reduced = xgb.XGBClassifier(
    n_estimators=100, max_depth=best_params["max_depth"], learning_rate=best_params["lr"],
    reg_lambda=best_params["lam"], eval_metric="logloss", random_state=0
)
model_reduced.fit(X_train[:, keep_idx], y_train)
acc_full = accuracy_score(y_test, model.predict(X_test))
acc_reduced = accuracy_score(y_test, model_reduced.predict(X_test[:, keep_idx]))

print(f"\nAccuracy, all {len(feature_names)} features:  {acc_full*100:.1f}%")
print(f"Accuracy, top {len(keep)} SHAP features: {acc_reduced*100:.1f}%")


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mean |SHAP value| per feature (impact on model output):
  Sex          1.3258
  Pclass       0.6834
  Fare         0.5030
  Age          0.4192
  Embarked     0.2491
  FamilySize   0.1507
  SibSp        0.0433
  Parch        0.0256
  IsAlone      0.0000

Dropping low-impact features: ['IsAlone']
Keeping: ['Sex', 'Pclass', 'Fare', 'Age', 'Embarked', 'FamilySize', 'SibSp', 'Parch']

Accuracy, all 9 features:  79.9%
Accuracy, top 8 SHAP features: 79.9%


## What the numbers actually show

- **Verification:** correlation 1.0000 between scratch and real `xgboost` predicted probabilities on a single, controlled boosting round — the gain and leaf-weight formulas are implemented correctly. They aren't bit-identical because the real library uses `base_score=0.5` by default and float32 internals even in "exact" tree-method mode.
- **Benchmark:** on this dataset, the gap between vanilla GBM and XGBoost-style second-order boosting is small (80.4% both, vs. 78.8% for a single tree and 79.3% for a random forest). With only 179 test examples, one flipped prediction moves accuracy by about 0.56 percentage points, so small differences here are close to noise — worth saying honestly rather than overclaiming a win.
- **SHAP:** `IsAlone` had essentially zero mean |SHAP value| — unsurprising, since it's a deterministic function of `FamilySize` (`FamilySize == 1`) and therefore redundant once that feature is already in the model. Dropping it cost no accuracy (79.9% to 79.9%), which is the actually useful finding: a simpler model, same performance, found systematically instead of by guessing.
